# Workflow:
- Load clean data
- Feature and Target
- Imputing
- Scaling and normalization
- Encoding
- Train Test Split
- Build and train 5 models
- Best Model Selection
- Save best model
- App Building
- Deployment

In [1]:
# Imports

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn import metrics
import warnings
warnings.filterwarnings("ignore")
print("import done")

import done


In [2]:
# Load clean data
data = pd.read_csv(r"C:\Users\Hp\Desktop\BIA\DATA SCIENCE PROJECTS\ds_project_1\data\cleaned_data.csv")
data.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Total Charges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,151.65


In [3]:
# backup 
df = data.copy()

In [4]:
# Encode target
df["Churn_Encoded"] = df["Churn"].map({"Yes":1, "No":0})
df.head(2)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Total Charges,Churn_Encoded
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,One year,No,Mailed check,56.95,1889.50,No,1889.50,0


In [5]:
# Feature and target
X = df.drop(columns = ["Churn", "Churn_Encoded"])
y = df["Churn_Encoded"]

# Numeric and Categorical Features
num_features = X.select_dtypes(include = "number").columns.to_list()
cat_features = X.select_dtypes(include = "object").columns.to_list()
print("Numerical Features: \n", num_features)
print("\nCategorical Features: \n", cat_features)

Numerical Features: 
 ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Total Charges']

Categorical Features: 
 ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


# Data Pre-processing

In [6]:
num_pipeline = Pipeline(steps = [
    ("imputer", SimpleImputer(strategy = "median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps = [
    ("imputer", SimpleImputer(strategy = "most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown = "ignore", sparse_output = False, dtype = np.int64, drop = "first"))
])

# Combine the pipeline
preprocessor = ColumnTransformer(transformers = [
    ("num_transformer", num_pipeline, num_features),
    ("cat_transformer", cat_pipeline, cat_features)
])

In [7]:
# Train-test-split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, stratify = y, random_state = 42)

In [8]:
# Models

models = {
    "LogisticRegression":(
        LogisticRegression(max_iter = 1000, solver="liblinear"),
        {"model__C":[0.01,0.1,1]}
    ),
    "RandomForest":(
        RandomForestClassifier(random_state = 42),
        {
            "model__n_estimators":[50,100,200],
            "model__max_depth":[None,5,10,20]
        }
    ),
    "AdaBoost":(
        AdaBoostClassifier(random_state = 42), 
        {
            "model__n_estimators":[50,100,200]
        }
    ),
    "XGB":(
        XGBClassifier(eval_metric = "logloss", random_state=42),
        {
            "model__n_estimators":[50,100,200],
            "model__learning_rate":[0.01,0.1]
        }
    ),
    "SVC":(
        SVC(probability= True),
        {
            "model__C":[0.01,0.1,1],
            "model__kernel": ["linear", "rbf", "polynomial"]
        }
    )
}

In [9]:
# Model training

results = []
best_model = None
best_model_name = None
best_cv_score = 0

for name,(model,params) in models.items():
    print(f"{name} is running...")
    pipe = Pipeline(steps = [
        ("processor", preprocessor),
        ("model", model)
    ])
    grid = GridSearchCV(estimator = pipe, param_grid = params, cv = 5, 
                       scoring = "f1", n_jobs = -1)
    grid.fit(X_train, y_train)


    # Prediction
    y_pred = grid.predict(X_test)

    # Evaluation Metrics
    accuracy = metrics.accuracy_score(y_test, y_pred)
    precision = metrics.precision_score(y_test, y_pred)
    recall = metrics.recall_score(y_test, y_pred)
    f1 = metrics.f1_score(y_test, y_pred)

    # Store result
    results.append({
        "Model":name,
        "CV_F1":grid.best_score_,
        "Accuracy":accuracy,
        "Precision":precision,
        "Recall": recall,
        "Test_F1": f1
    })

    if grid.best_score_ > best_cv_score:
        best_model = grid.best_estimator_
        best_model_name = name

print("Training is complete")

LogisticRegression is running...
RandomForest is running...
AdaBoost is running...
XGB is running...
SVC is running...
Training is complete


In [10]:
# Model Comparison

result_df = pd.DataFrame(results)
result_df = result_df.sort_values(by = "CV_F1", ascending = False)
result_df

,Model,CV_F1,Accuracy,Precision,Recall,Test_F1
0,LogisticRegression,0.597485,0.804116,0.654088,0.556150,0.601156
4,SVC,0.583973,0.787083,0.615625,0.526738,0.567723
2,AdaBoost,0.576048,0.804826,0.668942,0.524064,0.587706
3,XGB,0.571179,0.798439,0.646104,0.532086,0.583578
1,RandomForest,0.568073,0.805536,0.663399,0.542781,0.597059


In [11]:
# import
import joblib

In [12]:
# Save Model

BEST_MODEL_PATH = r"C:\Users\Hp\Desktop\BIA\DATA SCIENCE PROJECTS\ds_project_1\best_model.pkl"
joblib.dump(best_model, BEST_MODEL_PATH)
print(f"Best Trained Model is saved as {best_model_name}")
print("Location: ", BEST_MODEL_PATH)

Best Trained Model is saved as SVC
Location:  C:\Users\Hp\Desktop\BIA\DATA SCIENCE PROJECTS\ds_project_1\best_model.pkl
